# LAB 06: SIFT-Based Landmark Recognition & Geometric Localization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheBilalButt/lab-06-sift-landmark-recognition/blob/main/Lab6_SIFT_Landmark_Recognition.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/TheBilalButt/lab-06-sift-landmark-recognition)
[![Course](https://img.shields.io/badge/Course-Computer%20Vision-blueviolet)](https://github.com/TheBilalButt)
[![Python](https://img.shields.io/badge/Python-3.10%20%7C%203.11%20%7C%203.12%20%7C%203.13-blue?logo=python)](https://www.python.org/)
[![OpenCV](https://img.shields.io/badge/OpenCV-5.0.0-green?logo=opencv)](https://opencv.org/)

**Author:** Bilal Butt ([@TheBilalButt](https://github.com/TheBilalButt))  
**Institution:** Department of Computer Science / Artificial Intelligence  
**Repository:** [TheBilalButt / lab-06-sift-landmark-recognition](https://github.com/TheBilalButt/lab-06-sift-landmark-recognition)  

---

## 1. Problem Statement & Mathematical Foundation

A modern tourism information system requires automated recognition of architectural landmarks from unstructured visitor photographs subject to drastic viewpoint shifts, in-plane and out-of-plane rotations, scale changes, variable illumination, and occlusions.

### Mathematical Principles of SIFT (Scale-Invariant Feature Transform)
1. **Scale-Space Representation & Difference-of-Gaussians (DoG):**
   $$L(x, y, \sigma) = G(x, y, \sigma) * I(x, y)$$
   $$D(x, y, \sigma) = (G(x, y, k\sigma) - G(x, y, \sigma)) * I(x, y) = L(x, y, k\sigma) - L(x, y, \sigma)$$
2. **Sub-Pixel Localization & Edge Response Rejection:**
   $$\mathbf{x}_0 = - \left( \frac{\partial^2 D}{\partial \mathbf{x}^2} \right)^{-1} \frac{\partial D}{\partial \mathbf{x}}$$
   Using Hessian eigenvalue ratio to reject ill-conditioned 1D edges: $\frac{\text{Tr}(H)^2}{\text{Det}(H)} < \frac{(r+1)^2}{r}$ (where $r=10$).
3. **Orientation Assignment:**
   $$m(x, y) = \sqrt{(L(x+1, y) - L(x-1, y))^2 + (L(x, y+1) - L(x, y-1))^2}$$
   $$\theta(x, y) = \text{atan2}(L(x, y+1) - L(x, y-1), L(x+1, y) - L(x-1, y))$$
4. **128-Dimensional SIFT Descriptor:**
   $4 \times 4$ spatial grid of $8$-bin gradient histograms $\implies 4 \times 4 \times 8 = 128$ dimensions, normalized by Euclidean norm and thresholded at $0.2$.
5. **Lowe's Ratio Test ($k=2$ KNN):**
   $$\text{Accept match } m \iff \frac{\|d_q - d_{r1}\|}{\|d_q - d_{r2}\|} < 0.75$$
6. **Planar Homography & RANSAC Localization:**
   $$s \begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} = \begin{bmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & h_{33} \end{bmatrix} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

In [ ]:
# Setup dependencies and verify environment
import os
import sys
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt

print(f"OpenCV Version: {cv2.__version__}")
print(f"NumPy Version: {np.__version__}")
sift = cv2.SIFT_create()
print("SIFT Engine Initialized Successfully!")

## 2. Task 1: Landmark Image Database Construction

We maintain reference images of 5 world-famous landmarks plus 1 unknown non-database landmark:
- **Eiffel Tower** (Paris, France)
- **Taj Mahal** (Agra, India)
- **Great Wall of China** (Beijing, China)
- **Statue of Liberty** (New York, USA)
- **Colosseum** (Rome, Italy)
- **Big Ben / Palace of Westminster** (London, UK - Used as Non-Database Negative Sample)

In [ ]:
from src.sift_engine import SIFTEngine

engine = SIFTEngine()
database = engine.index_reference_database('data/landmarks')

print(f"Indexed {len(database)} landmarks in reference database:")
for lm, refs in database.items():
    print(f" - {lm}: {len(refs)} reference views")

# Display Database Gallery
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (lm, refs) in zip(axes, sorted(database.items())):
    rgb = cv2.cvtColor(refs[0]['bgr'], cv2.COLOR_BGR2RGB)
    ax.imshow(rgb)
    ax.set_title(lm.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Task 2: SIFT Feature Extraction & Scale-Space Keypoints

Each reference image is converted to grayscale, and SIFT keypoints with scale (circle radius) and orientation (radial line) are extracted along with 128-dimensional descriptors.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (lm, refs) in zip(axes, sorted(database.items())):
    ref = refs[0]
    kp_vis = engine.draw_keypoints(ref['bgr'], ref['keypoints'][:400], rich=True)
    ax.imshow(cv2.cvtColor(kp_vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{lm}\n{len(ref['keypoints'])} KPs", fontsize=10, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Task 3 & 4: Feature Matching & Landmark Recognition

We employ Brute-Force Matcher with Euclidean distance ($L_2$ norm), $k=2$ KNN matching, and Lowe's ratio test ($0.75$). Candidates are ranked based on reliable match count, subject to a minimum threshold ($T_{\text{match}} = 20$) and margin check to reject unknown landmarks.

In [ ]:
from src.feature_matcher import FeatureMatcher

matcher = FeatureMatcher(ratio_threshold=0.75, min_match_threshold=20)

# Test Query: Rotated Eiffel Tower
eiffel_bgr = database['eiffel_tower'][0]['bgr']
h, w = eiffel_bgr.shape[:2]
M = cv2.getRotationMatrix2D((w/2, h/2), 45, 0.85)
query_rot = cv2.warpAffine(eiffel_bgr, M, (w, h), borderMode=cv2.BORDER_REFLECT)

gray_q, kp_q, des_q = engine.extract_features(query_rot)
result = matcher.recognize_landmark(des_q, database)

print(f"Recognition Decision: {'RECOGNIZED' if result['is_recognized'] else 'REJECTED'}")
print(f"Predicted Landmark: {result['predicted_landmark']}")
print(f"Reliable Matches: {result['top_candidate']['best_match_count']}")

## 5. Task 5: Landmark Localization via RANSAC Homography

When sufficient reliable matches are established, we compute the homography matrix $H$ using RANSAC. The 4 bounding corners of the reference image are transformed into the query space and validated for convexity and area.

In [ ]:
from src.geometric_localizer import GeometricLocalizer

localizer = GeometricLocalizer(ransac_threshold=5.0, min_inliers=12)
top = result['top_candidate']

loc_res = localizer.estimate_homography_and_localize(
    query_rot, kp_q,
    top['best_ref_meta']['bgr'], top['best_ref_meta']['keypoints'],
    top['best_ref_matches']
)

print(f"Localization Success: {loc_res['success']}")
print(f"RANSAC Inliers: {loc_res['inliers_count']} / {top['best_match_count']} ({loc_res['inlier_ratio']*100:.1f}%)")

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(loc_res['annotated_image'], cv2.COLOR_BGR2RGB))
plt.title("Localized Landmark Bounding Polygon", fontsize=12, fontweight='bold')
plt.axis('off')
plt.show()

## 6. Task 6: Environmental Robustness & Performance Benchmark

We evaluate the recognition system under 6 comprehensive experimental conditions:
1. **Original Landmark** (Baseline)
2. **Rotated Landmark** (45° In-Plane Rotation)
3. **Resized Landmark** (0.5x Spatial Scale Reduction)
4. **Reduced Illumination** (0.35x Low-Light Attenuation)
5. **Different Viewpoint** (Novel Out-of-Plane Angular Perspective)
6. **Non-Database Landmark** (Rejection Validation via Big Ben)

### Empirical Results Summary Table

| Condition | Ground Truth | Predicted Landmark | Query KPs | Matches | Inliers | Decision | Latency |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Original Landmark** | `eiffel_tower` | `eiffel_tower` | `2,946` | `2,946` | `2,946` | **RECOGNIZED** | `989.7 ms` |
| **Rotated (45 deg)** | `eiffel_tower` | `eiffel_tower` | `2,517` | `1,234` | `1,227` | **RECOGNIZED** | `803.8 ms` |
| **Resized (0.5x Scale)** | `eiffel_tower` | `eiffel_tower` | `672` | `509` | `508` | **RECOGNIZED** | `260.8 ms` |
| **Low Brightness (0.35x)** | `eiffel_tower` | `eiffel_tower` | `760` | `718` | `717` | **RECOGNIZED** | `274.1 ms` |
| **Different Viewpoint** | `eiffel_tower` | `eiffel_tower` | `1,020` | `1,020` | `1,020` | **RECOGNIZED** | `333.7 ms` |
| **Non-Database Landmark** | `unknown` | `Unknown / Rejected` | `1,984` | `22` | `0` | **REJECTED** | `614.7 ms` |

### Robustness Evaluation Gallery

![Robustness Evaluation](figures/task6_robustness_evaluation.png)

### Latency and Matches Summary

![Metrics Summary](figures/task6_metrics_summary.png)

## 7. Discussion & Limitations

### System Strengths:
- **Exceptional Scale and Rotation Invariance:** SIFT accurately identified the landmark under 45° rotation (1,234 matches, 1,227 inliers) and 0.5x scaling (509 matches, 508 inliers).
- **Photometric Resilience:** Under extreme low-light conditions (0.35x brightness), gradient orientation ratios remained invariant, securing 718 matches with 717 inliers.
- **Precise Geometric Rejection:** For non-database queries (Big Ben), only 22 random matches were found, which RANSAC completely rejected (0 inliers, failing the 12-inlier threshold), preventing false positive classifications.

### Limitations & Future Scope:
- **Computational Complexity:** SIFT feature extraction and $O(N \times M)$ brute-force matching require 250-900ms per image. For a database of 10,000+ landmarks, Approximate Nearest Neighbors (FLANN) or Bag-of-Visual-Words (BoVW) indexing with inverted files would be necessary.
- **Repetitive Textures:** Symmetrical architectures with repeated arches can occasionally introduce 1-to-many match ambiguities requiring strict Lowe ratio testing.